# 02. 공유 encoder를 쓰는 toy multi-task trainer

목표: content와 motion branch가 하나의 encoder parameter를 함께 갱신하는 과정을 직접 구현하고 task weight의 역할을 관찰한다. 실제 MC-JEPA architecture를 재현하지 않는다.

In [ ]:
import random

rng = random.Random(13)
content_samples = [
    (value := rng.uniform(-1.0, 1.0), 1.5 * value)
    for _ in range(128)
]
motion_samples = [
    (delta := rng.uniform(-1.0, 1.0), 2.0 * delta)
    for _ in range(128)
]
len(content_samples), len(motion_samples)

In [ ]:
def train(flow_weight, epochs=120, learning_rate=0.08):
    # encoder weight는 공유하고 task별 head는 분리한다.
    encoder = 0.7
    content_head = 0.3
    flow_head = 0.3

    for _ in range(epochs):
        grad_encoder_content = 0.0
        grad_content_head = 0.0
        grad_encoder_flow = 0.0
        grad_flow_head = 0.0

        for feature, target in content_samples:
            error = content_head * encoder * feature - target
            grad_content_head += error * encoder * feature
            grad_encoder_content += error * content_head * feature

        for feature, target in motion_samples:
            error = flow_head * encoder * feature - target
            grad_flow_head += error * encoder * feature
            grad_encoder_flow += error * flow_head * feature

        grad_content_head /= len(content_samples)
        grad_encoder_content /= len(content_samples)
        grad_flow_head /= len(motion_samples)
        grad_encoder_flow /= len(motion_samples)

        content_head -= learning_rate * grad_content_head
        flow_head -= learning_rate * flow_weight * grad_flow_head
        encoder -= learning_rate * (
            grad_encoder_content + flow_weight * grad_encoder_flow
        )

    content_mse = sum(
        (content_head * encoder * x - y) ** 2
        for x, y in content_samples
    ) / len(content_samples)
    flow_mse = sum(
        (flow_head * encoder * x - y) ** 2
        for x, y in motion_samples
    ) / len(motion_samples)
    return {
        "flow_weight": flow_weight,
        "content_mse": content_mse,
        "flow_mse": flow_mse,
        "encoder": encoder,
    }

In [ ]:
results = [train(weight) for weight in (0.0, 0.01, 0.1, 1.0)]
for result in results:
    print(
        f"α={result['flow_weight']:<4} "
        f"content_mse={result['content_mse']:.6f} "
        f"flow_mse={result['flow_mse']:.6f} "
        f"encoder={result['encoder']:.3f}"
    )

assert results[2]["flow_mse"] < results[0]["flow_mse"]

## 해석과 확장

이 toy 문제는 두 head의 용량이 충분해 큰 task weight에서도 trade-off가 약할 수 있다. 실제 모델에서는 제한된 shared representation, 서로 다른 dataset, loss scale과 gradient 방향 때문에 한 task가 다른 task를 방해할 수 있다.

- 각 task의 gradient 부호와 크기를 기록한다.
- encoder는 하나의 scalar, head도 공유하도록 바꿔 의도적인 충돌을 만든다.
- combined loss와 batch alternation을 같은 update 수로 비교한다.
- raw loss가 아니라 gradient norm 기준으로 task weight를 조정하는 방법을 조사한다.